# Lab 2: Shrink and time

**AI in Space Applications II · after Lesson 14 · 30 minutes · guided**

**Goal.** Find out whether your cloud model fits VEGA-1's flight computer, and what one flipped bit does to it.

Requirement R3 says: the weights plus one tile's working memory fit in 250 MB, and 99% of scenes are processed within 2.27 seconds on the reference benchmark: ONNX Runtime, CPU, one thread, batch 1, after warm-up. A scene is 72 tiles of 512 × 512 pixels. Requirement R4 asks what one bit flip costs, and what you do about it.

**What you will have at the end:** an ONNX model in float32 and INT8, a benchmark with its conditions, a filled budget sheet, and a bit-flip test with a chosen mitigation. Capstone Phase 2 starts from this notebook.

**You need** `lab1_model.pt` from Lab 1 (it is in your Drive folder if you ran Lab 1 in Colab with Drive connected). If you lost it, the notebook trains a new one (a few minutes).

**In Colab.** Opened this from the course's *Open in Colab* link? First choose **File → Save a copy in Drive**, or your edits are lost when you close the tab. When the setup cell asks to connect Google Drive, allow it: the notebook keeps your models and results in the Drive folder `AI_in_Space_II_labs`, where later labs and sessions find them. The data files download by themselves.

In [ ]:
!pip -q install onnx==1.23.2 onnxruntime==1.31.0
import json
import os
import platform
import random
import shutil
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn

FAST = os.environ.get("VEGA1_FAST") == "1"        # set only by the course's own tests: a short run that checks the code
SEED = 0

KEEP_DIR = None                                   # your folder in Google Drive, when this runs in Colab
try:
    from google.colab import drive                # Colab only: Colab forgets files when a session ends, Drive does not
    drive.mount("/content/drive")
    KEEP_DIR = "/content/drive/MyDrive/AI_in_Space_II_labs"
    os.makedirs(KEEP_DIR, exist_ok=True)
    for name in os.listdir(KEEP_DIR):             # bring back what you saved in earlier labs and sessions
        if not os.path.exists(name) and os.path.isfile(os.path.join(KEEP_DIR, name)):
            shutil.copy2(os.path.join(KEEP_DIR, name), name)
    print("Your Drive folder:", KEEP_DIR, "| files found there:", ", ".join(sorted(os.listdir(KEEP_DIR))) or "none yet")
except ImportError:
    pass                                          # not in Colab: files stay beside the notebook
except Exception as e:
    print("Google Drive is not connected, so files last only for this session:", e)


def keep(*names):
    """Copy these files to your Drive folder, so the next session and the next lab find them."""
    if KEEP_DIR:
        for name in names:
            if os.path.exists(name):
                shutil.copy2(name, os.path.join(KEEP_DIR, name))
        print("Kept in your Drive folder:", ", ".join(names))


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python", platform.python_version(), "| PyTorch", torch.__version__, "| NumPy", np.__version__, "| device:", DEVICE)

In [ ]:
DATA_DIR = os.environ.get("VEGA1_DATA_DIR", ".")
DATA_URL = "https://github.com/Astronify-SpaceTech/ai-in-space-ii-labs/releases/download/data-v1/"   # the course's data files


def get_file(name):
    """Find a course data file: beside the notebook, else download it from the course's link, else upload it in Colab."""
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        return path
    if DATA_URL:
        import urllib.request
        try:
            print(f"downloading {name} ...")
            urllib.request.urlretrieve(DATA_URL + name, path + ".part")
            os.replace(path + ".part", path)
            return path
        except Exception as e:
            print(f"download failed ({e}); choose the file you downloaded from the lesson page instead")
    try:
        from google.colab import files       # Colab only: choose the file you downloaded from the lesson page
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"Put {name} beside this notebook (it is attached to the lab on the course page).")


d = np.load(get_file("vega1_train.npz"))
X = d["X"].astype(np.float32) * float(d["scale"]) + float(d["offset"])     # stored value x scale + offset = reflectance
y = d["y"].astype(np.int64)                      # 0 clear, 1 thick cloud, 2 thin cloud, 3 cloud shadow
region, patch = d["region"], d["patch"]
CLASSES = ["clear", "thick cloud", "thin cloud", "cloud shadow"]

clear_share = (y == 0).mean(axis=(1, 2))         # share of each chip that is usable ground
usable = clear_share >= 0.5                      # R2 counts chips with at least half their area usable
print(f"{len(X)} chips, {X.shape[1]} bands, {X.shape[2]} x {X.shape[3]} pixels, {len(np.unique(region))} regions, "
      f"{len(np.unique(patch))} patches")

In [ ]:
class Block(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class SmallUNet(nn.Module):
    """Input: (batch, 7, H, W) reflectance. Output: (batch, 4, H, W) class scores. H and W must divide by 4."""

    def __init__(self, bands=7, classes=4, widths=(16, 32, 64)):
        super().__init__()
        a, b, c = widths
        self.register_buffer("mean", torch.zeros(1, bands, 1, 1))
        self.register_buffer("std", torch.ones(1, bands, 1, 1))
        self.enc1, self.enc2, self.mid = Block(bands, a), Block(a, b), Block(b, c)
        self.dec2, self.dec1 = Block(c + b, b), Block(b + a, a)
        self.head = nn.Conv2d(a, classes, 1)

    def features(self, x):
        x = (x - self.mean) / self.std
        e1 = self.enc1(x)
        e2 = self.enc2(Fn.max_pool2d(e1, 2))
        m = self.mid(Fn.max_pool2d(e2, 2))
        return e1, e2, m

    def forward(self, x):
        e1, e2, m = self.features(x)
        d2 = self.dec2(torch.cat([Fn.interpolate(m, scale_factor=2.0, mode="nearest"), e2], dim=1))
        d1 = self.dec1(torch.cat([Fn.interpolate(d2, scale_factor=2.0, mode="nearest"), e1], dim=1))
        return self.head(d1)


def train_model(X_train, y_train, make_model=SmallUNet, epochs=None, seed=SEED):
    """Train a model on the given chips and return it. About a minute on a GPU, a few minutes on a CPU."""
    set_seed(seed)
    epochs = epochs or (1 if FAST else 6)
    model = make_model().to(DEVICE)
    model.mean.copy_(torch.tensor(X_train.mean(axis=(0, 2, 3))).view(1, -1, 1, 1))      # normalise with the training
    model.std.copy_(torch.tensor(X_train.std(axis=(0, 2, 3)) + 1e-6).view(1, -1, 1, 1))  # chips' own statistics
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    xt, yt = torch.from_numpy(X_train), torch.from_numpy(y_train)
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(xt))
        total = 0.0
        for k in range(0, len(order), 16):
            idx = order[k:k + 16]
            xb, yb = xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            if random.random() < 0.5:                     # a mirror flip: the same chip, seen another way
                xb, yb = xb.flip(-1), yb.flip(-1)
            logits = model(xb)
            step = yb.shape[-1] // logits.shape[-1]       # a model with a coarser output is scored on coarser labels
            loss = Fn.cross_entropy(logits, yb[:, ::step, ::step])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss) * len(idx)
        print(f"  epoch {epoch + 1}/{epochs}: loss {total / len(order):.3f} ({time.time() - t0:.0f} s)")
    return model.eval()


@torch.no_grad()
def predict(model, X_any, batch=32):
    """Class probabilities for every pixel, at the chip's own size: (chips, 4, H, W)."""
    out = []
    for k in range(0, len(X_any), batch):
        xb = torch.from_numpy(X_any[k:k + batch]).to(DEVICE)
        p = torch.softmax(model(xb), dim=1)
        if p.shape[-1] != xb.shape[-1]:
            p = Fn.interpolate(p, size=xb.shape[-2:], mode="nearest")
        out.append(p.cpu().numpy())
    return np.concatenate(out)

In [ ]:
def scores(model, X_test, y_test):
    """Cloud IoU (thick and thin cloud together), usable-area error in points, and pixel accuracy."""
    prob = predict(model, X_test)
    pred = prob.argmax(axis=1)
    pred_cloud, true_cloud = np.isin(pred, (1, 2)), np.isin(y_test, (1, 2))
    iou = (pred_cloud & true_cloud).sum() / max((pred_cloud | true_cloud).sum(), 1)
    area_error = np.abs(prob[:, 0].mean(axis=(1, 2)) - (y_test == 0).mean(axis=(1, 2))).mean() * 100
    return {"cloud_iou": float(iou), "usable_area_error_points": float(area_error),
            "pixel_accuracy": float((pred == y_test).mean())}


def region_split(region, share=0.25, seed=SEED):
    """Hold out whole regions: (train indices, test indices, the held-out regions)."""
    regions = np.random.default_rng(seed).permutation(np.unique(region))
    test_regions = regions[:max(1, int(len(regions) * share))]
    is_test = np.isin(region, test_regions)
    return np.where(~is_test)[0], np.where(is_test)[0], test_regions

## 1. Load your Lab 1 model

The checkpoint also holds the regions Lab 1 kept out, so you score on the same held-out ground.

In [ ]:
if os.path.exists("lab1_model.pt"):
    ck = torch.load("lab1_model.pt", map_location="cpu")
    model = SmallUNet(widths=tuple(ck["widths"]))
    model.load_state_dict(ck["state"])
    model.eval()
    test_regions = np.array(ck["test_regions"])
    is_test = np.isin(region, test_regions)
    train_idx, test_idx = np.where(~is_test)[0], np.where(is_test)[0]
    print("loaded lab1_model.pt; held-out regions:", ck["test_regions"])
else:
    print("lab1_model.pt not found: training a new model on the same region split (a few minutes)")
    train_idx, test_idx, test_regions = region_split(region)
    model = train_model(X[train_idx], y[train_idx])
base = scores(model, X[test_idx], y[test_idx])
print("held-out regions:", {k: round(v, 3) for k, v in base.items()})

## 2. VEGA-1's tile, and the flight graph

Your model learned on chips with 20 m pixels. VEGA-1's pixels are 4.75 m, so a VEGA-1 tile of 512 × 512 pixels, averaged in blocks of 4 × 4, has pixels of about 19 m: close to what the model was trained on. The **flight graph** does exactly that: average 4 × 4, then run the model. It is a design choice with a cost: the map comes back at a quarter of the camera's detail. Note it in your risks.

To time the flight graph you need a tile of the right size. The cell below makes one by repeating every pixel of a real chip 4 × 4: averaging it again gives back the chip exactly, so accuracy can be checked through the flight graph too.

In [ ]:
BIN = 4


class Flight(nn.Module):
    """VEGA-1's flight graph: a tile of 512 x 512 pixels at about 5 m, averaged 4 x 4, then the model."""

    def __init__(self, core):
        super().__init__()
        self.core = core

    def forward(self, x):
        return self.core(Fn.avg_pool2d(x, BIN))


def as_vega1_tile(chip):
    """A chip of 128 x 128 at 20 m, repeated 4 x 4 into a 512 x 512 tile (averaging it again gives the chip back)."""
    return np.repeat(np.repeat(chip, BIN, axis=-2), BIN, axis=-1)[None].astype(np.float32)


tile = as_vega1_tile(X[test_idx[0]])
flight = Flight(model).eval()
with torch.no_grad():
    back = Fn.avg_pool2d(torch.from_numpy(tile), BIN).numpy()
print("tile:", tile.shape, "| averaged back to the chip exactly:", np.allclose(back[0], X[test_idx[0]], atol=1e-6))

## 3. Core task, step 1: export to ONNX, and check it

Export the flight graph, then **check before you trust**: the ONNX model must give the same answers as PyTorch.

**Checkpoint.** The largest difference should be tiny, well under 0.001.

In [ ]:
import onnxruntime as ort
import logging
logging.getLogger("root").setLevel(logging.ERROR)


def export(m, path):
    torch.onnx.export(m.eval(), torch.from_numpy(tile), path, opset_version=17, input_names=["x"], output_names=["y"],
                      dynamo=False)
    return path


def session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 1           # the reference benchmark: one thread
    so.inter_op_num_threads = 1
    return ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])


export(flight, "flight_fp32.onnx")
s32 = session("flight_fp32.onnx")
with torch.no_grad():
    ref = flight(torch.from_numpy(tile)).numpy()
out = s32.run(None, {"x": ____})[0]           # run the ONNX model on the tile
max_diff = float(np.abs(____ - ____).max())   # the largest difference from PyTorch's answer
print(f"ONNX against PyTorch: largest difference {max_diff:.2e}")
assert max_diff < 1e-3

## 4. Core task, step 2: quantize to INT8

Static INT8 quantization with ONNX Runtime (Lesson 11): weights get one scale per output channel, and activations a scale learned from calibration tiles.

**Checkpoint.** The INT8 file is several times smaller. The held-out score should move by a point or two at most; if it moves more, look at your calibration tiles.

In [ ]:
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static


class Calibration(CalibrationDataReader):
    def __init__(self, idx):
        self.it = iter([{"x": as_vega1_tile(X[i])} for i in idx])

    def get_next(self):
        return next(self.it, None)


calib = np.random.default_rng(SEED).choice(train_idx, size=4 if FAST else 32, replace=False)
quantize_static("flight_fp32.onnx", "flight_int8.onnx", Calibration(calib), quant_format=QuantFormat.QDQ, per_channel=True,
                activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
s8 = session("flight_int8.onnx")


def onnx_scores(sess, idx):
    """Scores through the flight graph, on held-out chips turned into VEGA-1-sized tiles."""
    probs = []
    for i in idx:
        logits = sess.run(None, {"x": as_vega1_tile(X[i])})[0]
        p = np.exp(logits - logits.max(axis=1, keepdims=True))
        p = p / p.sum(axis=1, keepdims=True)
        if p.shape[-1] != X.shape[-1]:
            p = p.repeat(X.shape[-1] // p.shape[-1], axis=-2).repeat(X.shape[-1] // p.shape[-1], axis=-1)
        probs.append(p[0])
    prob = np.stack(probs)
    pred = prob.argmax(axis=1)
    yt = y[idx]
    pc, tc = np.isin(pred, (1, 2)), np.isin(yt, (1, 2))
    return {"cloud_iou": float((pc & tc).sum() / max((pc | tc).sum(), 1)),
            "usable_area_error_points": float(np.abs(prob[:, 0].mean(axis=(1, 2)) - (yt == 0).mean(axis=(1, 2))).mean() * 100)}


eval_idx = test_idx[:8] if FAST else test_idx
size32, size8 = os.path.getsize("flight_fp32.onnx"), os.path.getsize("flight_int8.onnx")
sc32, sc8 = onnx_scores(s32, eval_idx), onnx_scores(s8, eval_idx)
print(f"float32: {size32 / 1e6:.2f} MB, {sc32}")
print(f"INT8:    {size8 / 1e6:.2f} MB ({size32 / size8:.1f} times smaller), {sc8}")

## 5. Core task, step 3: the benchmark

The reference benchmark, exactly as R3 states it: one thread, batch 1, after 10 warm-up runs, every tile of a scene timed and added up, and the **99th percentile** of scene times reported. With a few dozen scenes, that percentile is close to the slowest scene you saw: a cautious reading, which is what a review board wants.

**Estimate first.** Your Lab 1 model is a full U-Net. Will 72 tiles tiles fit in 2.27 seconds?

In [ ]:
def benchmark(sess, scenes=None, tiles_per_scene=72, warmup=10):
    scenes = scenes or (2 if FAST else 30)
    for _ in range(warmup):
        sess.run(None, {"x": tile})
    times = []
    for _ in range(scenes * tiles_per_scene):
        t0 = time.perf_counter()
        sess.run(None, {"x": tile})
        times.append(time.perf_counter() - t0)
    t = np.array(times)
    scene_s = t.reshape(scenes, tiles_per_scene).sum(axis=1)
    p99 = float(np.percentile(____, ____))     # the 99th percentile of the scene times
    return {"tile_ms_p50": float(np.median(t) * 1000), "scene_s_mean": float(scene_s.mean()), "scene_s_p99": p99,
            "scenes": scenes, "runs": len(t)}


bench = {"lab1_fp32": benchmark(s32), "lab1_int8": benchmark(s8)}
for k, v in bench.items():
    print(k, {a: round(b, 3) for a, b in v.items()})

## 6. The compact reference model

If your model misses the time budget, you have three honest choices: shrink it and measure again, switch to this compact reference model and say so, or keep yours and record the fail. This model uses Lesson 11's depthwise-separable convolutions and stops at half the chip's detail: about a tenth of the U-Net's weights.

**Checkpoint.** It should fit the time budget with a wide margin, and lose a few points of accuracy. Is that a trade you would make? Write one sentence.

In [ ]:
def sep(cin, cout):
    """Depthwise-separable convolution (Lesson 11): a 3 x 3 filter per channel, then a 1 x 1 mix."""
    return nn.Sequential(nn.Conv2d(cin, cin, 3, padding=1, groups=cin, bias=False), nn.Conv2d(cin, cout, 1, bias=False),
                         nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class CompactNet(nn.Module):
    """The compact reference model. Input (batch, 7, H, W) reflectance; output class scores at H/2 x W/2."""

    def __init__(self, bands=7, classes=4, widths=(16, 32, 64)):
        super().__init__()
        a, b, c = widths
        self.register_buffer("mean", torch.zeros(1, bands, 1, 1))
        self.register_buffer("std", torch.ones(1, bands, 1, 1))
        self.enc1 = nn.Sequential(nn.Conv2d(bands, a, 3, padding=1, bias=False), nn.BatchNorm2d(a), nn.ReLU(inplace=True))
        self.enc2 = sep(a, b)
        self.mid = nn.Sequential(sep(b, c), sep(c, c))
        self.dec = sep(c + b, b)
        self.head = nn.Conv2d(b, classes, 1)

    def forward(self, x):
        x = (x - self.mean) / self.std
        e1 = self.enc1(x)
        e2 = self.enc2(Fn.max_pool2d(e1, 2))
        m = self.mid(Fn.max_pool2d(e2, 2))
        return self.head(self.dec(torch.cat([Fn.interpolate(m, scale_factor=2.0, mode="nearest"), e2], dim=1)))


compact = train_model(X[train_idx], y[train_idx], make_model=CompactNet)
export(Flight(compact), "compact_fp32.onnx")
quantize_static("compact_fp32.onnx", "compact_int8.onnx", Calibration(calib), quant_format=QuantFormat.QDQ, per_channel=True,
                activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
c32, c8 = session("compact_fp32.onnx"), session("compact_int8.onnx")
n_params = {"lab1": sum(p.numel() for p in model.parameters()), "compact": sum(p.numel() for p in compact.parameters())}
csc32, csc8 = onnx_scores(c32, eval_idx), onnx_scores(c8, eval_idx)
bench["compact_fp32"], bench["compact_int8"] = benchmark(c32), benchmark(c8)
print("weights:", n_params)
for k in ("compact_fp32", "compact_int8"):
    print(k, {a: round(b, 3) for a, b in bench[k].items()})
print("compact float32:", csc32, "| compact INT8:", csc8)

## 7. Memory: weights plus one tile's working memory

Working memory is what a tile needs while it flows through the network: the input, the feature maps kept for the skip connections, and the largest layer's input and output at the same moment. The cell counts them with hooks on the PyTorch model. It is an estimate, and a cautious one.

In [ ]:
def working_memory_bytes(core):
    sizes = []
    hooks = [m.register_forward_hook(lambda _m, _i, o: sizes.append(o.numel() * 4))
             for m in core.modules() if len(list(m.children())) == 0]
    with torch.no_grad():
        Flight(core)(torch.from_numpy(tile))
    for h in hooks:
        h.remove()
    pairs = max(a + b for a, b in zip(sizes, sizes[1:]))
    skips = sorted(sizes, reverse=True)[:2]
    return tile.nbytes + pairs + sum(skips)


mem = {}
for name, core, path in (("lab1_fp32", model, "flight_fp32.onnx"), ("lab1_int8", model, "flight_int8.onnx"),
                         ("compact_fp32", compact, "compact_fp32.onnx"), ("compact_int8", compact, "compact_int8.onnx")):
    w, wm = os.path.getsize(path), working_memory_bytes(core)
    mem[name] = {"weights_mb": w / 1e6, "working_mb": wm / 1e6, "total_mb": (w + wm) / 1e6}
    print(name, {k: round(v, 2) for k, v in mem[name].items()}, "against", 250, "MB")

## 8. Fill the budget sheet

A margin is the share of the budget left over: (budget − value) ÷ budget. Choose the model you will carry into the capstone: `CHOICE` below. Copy these lines into the budget sheet template.

In [ ]:
BUDGET_MB, BUDGET_S = 250.0, 2.27
CHOICE = "____"                  # the model you carry forward: lab1_fp32, lab1_int8, compact_fp32 or compact_int8
mem_key = CHOICE
t99 = bench[CHOICE]["scene_s_p99"]
sheet = {
    "memory": {"budget_mb": BUDGET_MB, "value_mb": mem[mem_key]["total_mb"], "margin": (BUDGET_MB - mem[mem_key]["total_mb"]) / BUDGET_MB},
    "time": {"budget_s": BUDGET_S, "value_s": t99, "margin": (BUDGET_S - t99) / BUDGET_S,
             "conditions": "ONNX Runtime CPU, 1 thread, batch 1, 10 warm-up runs, %d scenes of 72 tiles" % bench[CHOICE]["scenes"]},
}
for k, v in sheet.items():
    print(k, v, "->", "within budget" if v["margin"] >= 0 else "OVER BUDGET")

## 9. Your turn: flip one bit

A particle flips the top exponent bit (bit 30) of one float32 weight in the first layer. Then the same layer is stored as INT8 and a particle flips a sign bit. Predict both results before you run the cell.

In [ ]:
def flip_float32_top_exponent(w):
    """Flip bit 30 of a float32 number: the top bit of its exponent."""
    return float((np.array([w], dtype=np.float32).view(np.uint32) ^ np.uint32(1 << 30)).view(np.float32)[0])


def flip_int8_sign(q):
    """Flip bit 7 of an INT8 number: its sign bit in two's complement."""
    return int((np.array([q], dtype=np.int8).view(np.uint8) ^ np.uint8(0x80)).view(np.int8)[0])


def with_weight(core, layer, index, value):
    """A copy of the model with one weight replaced."""
    import copy
    m = copy.deepcopy(core)
    w = dict(m.named_parameters())[layer]
    with torch.no_grad():
        w.view(-1)[index] = value
    return m.eval()


target = CHOICE.split("_")[0]
core = compact if target == "compact" else model
layer = "enc1.0.weight"
w = dict(core.named_parameters())[layer].detach().view(-1)
index = int(torch.argmax(w.abs()))
w0 = float(w[index])
scale = float(w.abs().max()) / 127              # the layer's INT8 step (symmetric, one scale for this channel's largest weight)
q0 = int(round(w0 / scale))
q1 = flip_int8_sign(q0)
flipped = {"float32": with_weight(core, layer, index, flip_float32_top_exponent(w0)),
           "int8": with_weight(core, layer, index, q1 * scale)}
clean = scores(core, X[eval_idx], y[eval_idx])["usable_area_error_points"]
flip = {}
for k, m in flipped.items():
    with torch.no_grad():
        flip[k] = scores(m, X[eval_idx], y[eval_idx])["usable_area_error_points"]


def shown(v):
    return f"{v:.1f} points" if np.isfinite(v) else "no map at all (the network's outputs are not numbers)"


print(f"weight {w0:.4f}: float32 flip -> {flip_float32_top_exponent(w0):.3e}; INT8 {q0} -> {q1}")
print(f"usable-area error: clean {clean:.1f} points | float32 flip: {shown(flip['float32'])} | INT8 flip: {shown(flip['int8'])}")

**Choose a mitigation** (Lesson 13) and test it. The reference run uses a **range check**: record every layer's range on the ground, and before each scene replace any weight outside it with zero and raise a flag. Your choice may be different: INT8 storage, voting with three copies (and three times the energy), or reloading from protected memory.

In [ ]:
ranges = {n: (float(p.min()), float(p.max())) for n, p in core.named_parameters()}


def range_check(m):
    """Zero every weight outside its recorded range, and report how many were caught."""
    caught = 0
    with torch.no_grad():
        for n, p in m.named_parameters():
            lo, hi = ranges[n]
            bad = (p < lo) | (p > hi) | ~torch.isfinite(p)
            caught += int(bad.sum())
            p[bad] = 0.0
    return caught


guarded = flipped["____"]             # the damaged model to protect
caught = ____(guarded)                # apply your mitigation
after = scores(guarded, X[eval_idx], y[eval_idx])["usable_area_error_points"]
print(f"range check caught {caught} weight(s); usable-area error {after:.1f} points (clean {clean:.1f}); "
      f"R4 allows a loss of 2 points -> {'PASS' if after - clean <= 2 else 'MISS'}")

## 10. Check, then break, and save

**Check:** the benchmark above used one thread and a warm-up. **Break:** time the first ten runs of a fresh session with no warm-up and compare with the median. Then save your results for the capstone.

In [ ]:
cold = session("compact_fp32.onnx")
first = []
for _ in range(10):
    t0 = time.perf_counter()
    cold.run(None, {"x": tile})
    first.append((time.perf_counter() - t0) * 1000)
print(f"first run {first[0]:.1f} ms; median of the next nine {np.median(first[1:]):.1f} ms; after warm-up "
      f"{bench['compact_fp32']['tile_ms_p50']:.1f} ms")
results = {"choice": CHOICE, "params": n_params, "scores": {"lab1_fp32": sc32, "lab1_int8": sc8, "compact_fp32": csc32,
           "compact_int8": csc8}, "benchmark": bench, "memory": mem, "budget_sheet": sheet,
           "bit_flip": {"clean": clean, "float32": flip["float32"] if np.isfinite(flip["float32"]) else None, "int8": flip["int8"],
                 "range_check": after, "caught": caught},
           "sizes_mb": {"lab1_fp32": size32 / 1e6, "lab1_int8": size8 / 1e6}}
with open("lab2_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=1)
torch.save({"state": compact.state_dict(), "widths": (16, 32, 64), "seed": SEED,
            "test_regions": sorted(int(r) for r in test_regions)}, "compact_model.pt")
print("saved lab2_results.json and compact_model.pt")
keep("lab2_results.json", "compact_model.pt", "flight_fp32.onnx", "flight_int8.onnx", "compact_fp32.onnx", "compact_int8.onnx")

## Stretch (optional, not graded)

- **Passes and the flood-tile wait.** With Skyfield, load a real sun-synchronous satellite's orbit data from CelesTrak and list its passes over a ground station of your choice for a week, above a 10° elevation mask. What is the longest gap? That is the budget sheet's reported line.
- **A second runtime.** Load `compact_fp32.onnx` with OpenVINO and time it the same way. Same model, same chip: is it the same speed?